# Fold illusions: one picture flat, another folded

A sheet of paper that shows one picture when it lies flat, and a different picture once it's folded into a crane.

This builds on [`fold_map.ipynb`](fold_map.ipynb), which works out where every visible point of a folded model comes from on the sheet. The printed side is a single image seen two ways:

- **Flat:** the whole sheet, as is.
- **Folded:** only the visible regions (about a quarter of the sheet for the crane), cut up, rotated, and rearranged.

The hidden three quarters only have to serve the flat picture. The visible quarter has to serve both, and that's the whole challenge.

**Two approaches here:**

1. **Composite (no AI, runs anywhere):** the flat picture in hidden areas and the folded pictures in visible ones. Folded looks perfect; flat is covered in shards. A baseline that makes the problem visible.
2. **Generate both at once (needs a GPU):** let an image generator draw the flat picture and the folded pictures together from text prompts, so it can choose content that works both ways. This adapts [Visual Anagrams](https://dangeng.github.io/visual_anagrams/) (Geng, Park, Owens, CVPR 2024) to folding.

It works for the crane and any other flat-folded model that shows only one side of the paper.

## Setup

Clones [rath-hy/folded-illusions](https://github.com/rath-hy/folded-illusions/tree/main/fold-map) on Colab, or uses the local files if you run this from that folder.

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "fold-map"
if not os.path.exists("illusion.py"):
    if not os.path.exists("folded-illusions"):
        # The repo is private. On Colab, add a GitHub token as a secret named GITHUB_TOKEN (key icon, left sidebar).
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = f"https://{token}@github.com/rath-hy/folded-illusions.git"
        subprocess.run(["git", "clone", "-q", "--depth", "1", url], check=True)
    os.chdir(os.path.join("folded-illusions", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "directions", "decal", "illusion", "story"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from fold_map import FoldedModel
from decal import test_picture
from illusion import FoldViews, canvas_picture, composite, plot_result, printable, square_picture

## Load a model

The crane by default. To use another model, export its crease pattern and folded state from Flat-Folder (**Export → cp** and **Export → state**; see `fold_map.ipynb`) and upload both.

In [ ]:
CREASE_PATTERN = "004_traditional_Crane_cp.fold"
FOLDED_STATE = "004_traditional_Crane_state.fold"
UPLOAD_MODEL = False     # True: upload your own crease pattern and folded state (Colab)

if UPLOAD_MODEL:
    from google.colab import files
    uploaded = list(files.upload())
    FOLDED_STATE = next(name for name in uploaded if "state" in name.lower())
    CREASE_PATTERN = next(name for name in uploaded if name != FOLDED_STATE)

model = FoldedModel(CREASE_PATTERN, FOLDED_STATE)

## The views

`FoldViews` turns the fold map into pixel maps between the printed side of an R × R sheet and two folded **canvases** of the same size and scale: the folded model seen from the front and from the back, centered, on a plain background.

Folding moves paper rigidly, so each folded canvas is a rearrangement of the visible sheet pixels. Rotating a pixel grid by angles like 22.5° can't be exactly one-to-one, so a small fraction of canvas pixels share a sheet pixel. `unique` below is how close each view is to a pure rearrangement.

The generator works at 64 × 64 first, then 256 × 256. At 64, the folded crane covers only a few hundred pixels, so the folded picture's overall layout is decided at thumbnail size; detail comes in the second stage.

In [ ]:
views64, views256 = FoldViews(model, 64), FoldViews(model, 256)
print(f"printed side: {views256.printed_side}")
for name, v in (("64", views64), ("256", views256)):
    print(name, v.stats())

from fold_map import PALETTE     # hidden, visible from the front, visible from the back

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
v = views256
axes[0].imshow(PALETTE[v.visible["front"] * 1 + v.visible["back"] * 2])
axes[0].set_title("printed side: tan shows on the front, blue on the back")
for ax, view in zip(axes[1:], ("front", "back")):
    ax.imshow(np.where(v.paper[view][..., None], PALETTE[1 if view == "front" else 2], 0.95))
    ax.set_title(f"{view} canvas")
for ax in axes: ax.axis("off")
plt.show()

## 1. Composite (no AI)

Pick a flat picture and front and back pictures, or leave them as `None` for test pictures. Each folded picture is cropped to cover the folded model; the flat picture is cropped square.

The flat view and both folded views below come from **folding the sheet independently**, using the same simulator as the decal, so they double as a check on the views.

In [ ]:
FLAT_PICTURE = None      # path to an image, or None for a test picture
FRONT_PICTURE = None
BACK_PICTURE = None

load = lambda path, letter, color, size=(1300, 1000): Image.open(path) if path else test_picture(letter, color, size)
flat = square_picture(load(FLAT_PICTURE, "A", (60, 140, 90), (1000, 1000)), 256)
front = canvas_picture(views256, load(FRONT_PICTURE, "F", (196, 120, 60)), "front")
back = canvas_picture(views256, load(BACK_PICTURE, "B", (52, 98, 168)), "back")

sheet = composite(views256, flat, front, back)
plot_result(views256, sheet, "composite");

Folded, the pictures come out exactly. Flat, the picture has to compete with sharp, rotated shards of the folded ones over a quarter of its area. With two fixed, unrelated pictures, that conflict is unavoidable: each visible pixel has one color, and it can't match both. To do better, the pictures themselves have to cooperate.

## 2. Generate both at once

### How it works

A diffusion model makes an image by starting from noise and removing a little at each step, guided by a text prompt. [Visual Anagrams](https://dangeng.github.io/visual_anagrams/) makes one image that reads as different things from different "views" (flips, rotations, jigsaw rearrangements) like this:

1. At each step, show the model the current noisy image through each view, each with its own prompt.
2. Each view gets a prediction of the noise to remove.
3. Map each prediction back into the original image's layout and average them, then take the step.

This works when every view is a **rearrangement of pixels**: rearranged noise is still noise, so the model sees an ordinary noisy image in every view. Folding is a rearrangement (almost exactly; see `unique` above), so the folded crane can be one of the views. That's also why the model has to work on actual pixels: [DeepFloyd IF](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) generates 64 × 64 images and then upsamples them to 256 × 256 with a second model. Stable Diffusion works in a compressed latent space, where rearranging pixels doesn't correspond to rearranging anything the model sees.

**Adapting it to folding:**

- **Three views:** the flat sheet (prompt A), the folded front (prompt B), and the folded back (prompt C).
- **A plain backdrop.** Around the folded crane there's no paper. Those canvas pixels are held to a plain backdrop, re-noised to match each step as in inpainting, so the model draws B and C as crane-shaped pictures on a plain background.
- **A per-pixel blend.** Hidden sheet pixels follow only the flat view. Visible pixels blend the flat and folded predictions; `FOLD_WEIGHT` sets the balance.
- **Both stages.** Stage 2 (64 → 256) uses the same views, with the upscaled stage-1 sheet as its guide, seen through each view too.

### Before running

1. Use a **GPU runtime** (Runtime → Change runtime type; a T4 works).
2. On Hugging Face, accept the license for [DeepFloyd/IF-I-M-v1.0](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) and [DeepFloyd/IF-II-M-v1.0](https://huggingface.co/DeepFloyd/IF-II-M-v1.0), and have an [access token](https://huggingface.co/settings/tokens) ready.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece
from huggingface_hub import notebook_login
notebook_login()

### Prompts

Some tips from Visual Anagrams that apply here:

- **Share a style** across all prompts ("an oil painting of ...", "a watercolor of ...", "a lithograph of ..."). Painterly styles hide the seams where the two pictures compete.
- **Keep the flat picture tolerant of fragments.** Busy scenes and textures (a crowd, a forest, a mosaic, a city at night) absorb the folded picture's shards better than a single clean object.
- **Make the folded subject simple and bold.** It's drawn at thumbnail size in stage 1 and has to read on a crane-shaped patch: an animal, a face, a flower.

In [ ]:
STYLE = "an oil painting of "
FLAT_PROMPT = STYLE + "a crowded city street at night"
FRONT_PROMPT = STYLE + "a red fox, on a plain light background"
BACK_PROMPT = STYLE + "a white owl, on a plain light background"

### Encode the prompts

DeepFloyd's text encoder (T5-XXL) is about 9.5 GB, too big to keep in Colab's RAM next to everything else. So it's loaded straight onto the GPU, used once to turn the prompts into embeddings, and deleted. **Rerun this cell whenever you change the prompts.** The first run downloads about 11 GB; after that it loads from cache in about a minute.

In [ ]:
import gc, torch
from transformers import T5EncoderModel
from diffusers import IFPipeline, IFSuperResolutionPipeline

# The safety checker and watermarker aren't used by our sampler, and the safety checker
# doesn't load with current versions of transformers, so skip them.
skip = dict(safety_checker=None, feature_extractor=None, watermarker=None)

text_encoder = T5EncoderModel.from_pretrained("DeepFloyd/IF-I-M-v1.0", subfolder="text_encoder", variant="fp16",
                                              torch_dtype=torch.float16, device_map="cuda")
encoder = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=text_encoder, unet=None, **skip)
prompt_embeds, negative_embeds = encoder.encode_prompt([FLAT_PROMPT, FRONT_PROMPT, BACK_PROMPT])

del encoder, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("embeddings:", tuple(prompt_embeds.shape))

### Load the image models

The two image models are small (about 1 GB each at half precision) and live on the GPU. This only needs to run once per session.

In [ ]:
if "stage_1" not in globals():
    stage_1 = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=None, variant="fp16",
                                         torch_dtype=torch.float16, **skip).to("cuda")
    stage_2 = IFSuperResolutionPipeline.from_pretrained("DeepFloyd/IF-II-M-v1.0", text_encoder=None, variant="fp16",
                                                        torch_dtype=torch.float16, **skip).to("cuda")

### Generate

- `FOLD_WEIGHT`: how much the visible regions follow the folded pictures (1) vs. the flat one (0). Higher makes the folded crane cleaner and the flat picture more shattered.
- `GUIDANCE`: how strongly each stage follows the prompts (stage 1, stage 2).
- `SEED`: try several; like Visual Anagrams, some seeds work much better than others.

In [ ]:
from illusion import generate

FOLD_WEIGHT = 0.5
GUIDANCE = (7.0, 4.0)
STEPS = (50, 40)
SEED = 0

sheet64, sheet256 = generate(stage_1, stage_2, views64, views256, (negative_embeds, prompt_embeds),
                             guidance=GUIDANCE, steps=STEPS, fold_weight=FOLD_WEIGHT, seed=SEED)
plot_result(views64, sheet64, "stage 1 (64 × 64)");
plot_result(views256, sheet256, "stage 2 (256 × 256)");

### Print it

256 × 256 is coarse for an 8-inch square (about 32 dpi), but enough to judge the illusion. The saved image is upscaled smoothly to 2400 pixels. Print the image at **actual size**, cut the square, and fold the crane with the **printed side out**. Use the corner letters in the preview to orient it: for the crane, A becomes the tail, C the head, and B and D the wing tips. The letters aren't printed, so mark a corner lightly in pencil if it helps.

In [ ]:
name = f"illusion_seed{SEED}.png"
printable(sheet256).save(name, dpi=(300, 300))
print("saved", name)
try:
    from google.colab import files
    files.download(name)
except ImportError:
    pass

## Notes and next steps

- **The conflict is real.** Every visible pixel is shared, so there's always a trade-off, set by `FOLD_WEIGHT`. The generator's job is to find content where the trade-off doesn't show.
- **Resolution.** The folded picture is decided at 64 × 64, where the crane covers only a few hundred pixels. Starting stage 1 from a larger crane canvas, or an extra upsampling stage after 256, would sharpen things.
- **Better than averaging.** Visual Anagrams averages noise predictions; later work on these illusions explores other ways to combine views, which might handle the shared pixels better.
- **More illusions with the same machinery:** hidden pictures (prompt only the hidden region), the back of the crane continuing the front, or the flat sheet as a scene that contains the crane's picture as an object.

## References

- D. Geng, I. Park, A. Owens. [Visual Anagrams: Generating Multi-View Optical Illusions with Diffusion Models](https://dangeng.github.io/visual_anagrams/). *CVPR*, 2024.
- DeepFloyd. [IF](https://github.com/deep-floyd/IF): a pixel-space cascaded text-to-image diffusion model, 2023.
- C. Saharia et al. Photorealistic Text-to-Image Diffusion Models with Deep Language Understanding (Imagen). *NeurIPS*, 2022. The cascaded pixel-space design IF follows.
- A. Lugmayr et al. [RePaint](https://arxiv.org/abs/2201.09865): Inpainting using Denoising Diffusion Probabilistic Models. *CVPR*, 2022. Holding known pixels at the current noise level, used here for the backdrop.
- H. A. Akitaya, E. D. Demaine, J. S. Ku. [Computing Flat-Folded States](https://erikdemaine.org/papers/FlatFolder_OSME2024/). *OSME*, 2024. Flat-Folder, which provides the folded state.